# B3 · Gruplara ayırıp her grubun ortalamasını almak (groupby)

**Bugünün tek fikri:** tabloyu bir sütundaki değerlere göre **gruplara ayır**, her grubun içinde **bir hesap** yap (ortalama, sayı…).

Gündelik benzetme: sınıftaki sınav kâğıtlarını şubelere göre üç yığına ayırıyorsun, sonra her yığının ortalamasını alıyorsun. Ayır → hesapla → sonuçları yan yana koy.

B2'nin kapanışındaki sorunu bugün cevaplıyoruz: *GFS hangi mevsimde, hangi saatte en çok şaşıyor; hafif rüzgârda tutarlı mı?*

**Kural:** her kod hücresinden önce altındaki kutuya tek cümle yaz, sonra çalıştır.

## 1. Oyuncak tablo

6 satır. `hata` = gfs − ölçüm (B2'deki gibi; eksi ise GFS az tahmin etmiş).

In [12]:
import pandas as pd

oyuncak = pd.DataFrame({
    "mevsim": ["kis", "yaz", "kis", "yaz", "yaz", "ilkbahar"],
    "hata":   [-1,    -6,    -3,    -4,    -5,    -2],
})
oyuncak

,mevsim,hata
0,kis,-1
1,yaz,-6
2,kis,-3
3,yaz,-4
4,yaz,-5
5,ilkbahar,-2


✍️ *Bu hücre ne yapıyor (tek cümle):*
Bu hücre henüz bir şey ypamadı sadece tablo oluşturdu

### 1a. Önce elle: düz `for` döngüsü

groupby'ın içinde ne olduğunu görmek için aynı işi önce döngüyle yapıyoruz. Her mevsim için: o mevsimin satırlarını maskeyle seç (B1), `hata` ortalamasını al.

`.unique()` sütundaki farklı değerleri bir kez verir: `kis, yaz, ilkbahar`.

In [13]:
for m in oyuncak["mevsim"].unique():
    secili = oyuncak[oyuncak["mevsim"] == m]
    print(m, secili["hata"].mean())

kis -2.0
yaz -5.0
ilkbahar -2.0


✍️ *Bu hücre ne yapıyor (tek cümle):*
Karmaşık verileri Tek bir başlık altında toplayıp ort aldı

🧮 *Çalıştırmadan önce kafadan hesapla: kış kaç çıkmalı, yaz kaç?*

### 1b. Aynı iş tek satırda: `groupby`

Okuması soldan sağa: **mevsime göre grupla** → **`hata` sütununu al** → **ortalamasını hesapla**.

```
tablo.groupby("grup_sütunu")["hesap_sütunu"].mean()
```

In [24]:
oyuncak.groupby("mevsim")["hata"].mean()

mevsim
ilkbahar   -2.0
kis        -2.0
yaz        -5.0
Name: hata, dtype: float64

✍️ *Bu hücre ne yapıyor (tek cümle):*
For loop ile yaptığımız işi groupby ile yaptık ort aldık

🔍 *1a'daki döngüyle aynı sayılar mı? Sıra neden farklı?*

### 1c. Ortalama yerine başka hesap

`.mean()` yerine `.count()` (kaç satır), `.max()`, `.min()` de yazılır. Kalıp aynı, yalnız sondaki değişiyor.

In [15]:
oyuncak.groupby("mevsim")["hata"].count()

mevsim
ilkbahar    1
kis         2
yaz         3
Name: hata, dtype: int64

✍️ *Bu hücre ne yapıyor (tek cümle):*
For loop ile yaptığımız işi groupby ile yaptık kaç tane budluk

### 1d. Sonucu sıralamak

groupby'ın sonucu bir Series (B1b). B2'deki `sort_values` ona da uyar; Series'te sütun adı vermene gerek yok.

In [16]:
oyuncak.groupby("mevsim")["hata"].mean().sort_values()

mevsim
yaz        -5.0
ilkbahar   -2.0
kis        -2.0
Name: hata, dtype: float64

✍️ *Bu hücre ne yapıyor (tek cümle):*
ortalamalara göre sıraladık

## 2. Gerçek veri

İlk hücre B2'de senin ürettiğin sütunları yeniden ekliyor (yeni defter, tablo sıfırdan yükleniyor). Bunu Pezo yazdı, sadece çalıştır.

In [17]:
from ruzgar.data import load_aligned
veri = load_aligned()

mevsim_sozluk = {12: "kis", 1: "kis", 2: "kis",
                 3: "ilkbahar", 4: "ilkbahar", 5: "ilkbahar",
                 6: "yaz", 7: "yaz", 8: "yaz",
                 9: "sonbahar", 10: "sonbahar", 11: "sonbahar"}

veri["mutlak_hata"] = veri["hata"].abs()
veri["yerel_saat"] = (veri["time"].dt.hour + 3) % 24
veri["mevsim"] = veri["time"].dt.month.map(mevsim_sozluk)
veri["sinif"] = pd.cut(veri["speed_kn_obs"], bins=[0, 10, 15, 20, 100],
                       labels=["hafif", "orta", "iyi", "kuvvetli"], right=False)
veri.head()

,time,speed_kn_gfs,dir_deg_gfs,gust_kn_gfs,station,speed_kn_obs,dir_deg_obs,gust_kn_obs,hata,mutlak_hata,yerel_saat,mevsim,sinif
0,2024-03-01 01:00:00+00:00,1.8,139,2.1,LTBJ,2.0,NaN,NaN,-0.2,0.2,4,ilkbahar,hafif
1,2024-03-01 02:00:00+00:00,2.0,287,1.6,LTBJ,5.0,150.0,NaN,-3.0,3.0,5,ilkbahar,hafif
2,2024-03-01 03:00:00+00:00,1.6,346,1.7,LTBJ,2.0,160.0,NaN,-0.4,0.4,6,ilkbahar,hafif
3,2024-03-01 04:00:00+00:00,0.4,297,0.4,LTBJ,6.0,350.0,NaN,-5.6,5.6,7,ilkbahar,hafif
4,2024-03-01 05:00:00+00:00,0.9,207,0.6,LTBJ,4.0,350.0,NaN,-3.1,3.1,8,ilkbahar,hafif


### Alıştırma 1 · GFS hangi mevsimde en çok şaşıyor?

Mevsime göre grupla, `mutlak_hata` ortalamasını al, küçükten büyüğe sırala.

In [25]:
# 1b + 1d kalıbı: veri.groupby(...)[...].mean().sort_values()
veri.groupby("mevsim")["mutlak_hata"].mean().sort_values()

mevsim
kis         3.596846
ilkbahar    3.893794
sonbahar    4.399489
yaz         5.669360
Name: mutlak_hata, dtype: float64

✍️ *En kötü mevsim hangisi? Sence neden (ipucu: yazın İzmir'de hangi rüzgâr eser)?*
En kötü mevsimden kastın hata oranıysa yaz mevsimi

### Alıştırma 2 · Az mı tahmin ediyor, çok mu?

Aynı gruplama, ama bu sefer **işaretli** `hata` ortalaması. Eksi = GFS az tahmin ediyor.

In [26]:
# Alıştırma 1'deki satırı kopyala, yalnız sütun adını değiştir
veri.groupby("mevsim")["hata"].mean().sort_values()

mevsim
yaz        -5.358577
sonbahar   -3.708101
ilkbahar   -2.980794
kis        -2.347727
Name: hata, dtype: float64

✍️ *Dört mevsimin işareti aynı mı? 1'deki mutlak hatayla ne kadar yakın?*
hepsi negatif olduğu için gfs sanırım daha düşük yani tahmin gerçeğinden düşük

### Alıştırma 3 · Günün hangi saatinde?

`yerel_saat`'e göre grupla, `mutlak_hata` ortalaması. Bu sefer sıralama yerine saat sırasıyla bak (groupby zaten grup adına göre sıralı verir).

In [27]:
# grup sütunu değişiyor, gerisi aynı
veri.groupby("yerel_saat")["mutlak_hata"].mean().sort_values()

yerel_saat
3     3.874180
2     3.913204
1     3.945650
4     3.945855
5     3.993961
6     4.046004
0     4.171853
7     4.289959
23    4.474898
8     4.489344
19    4.561475
18    4.590574
20    4.601332
17    4.648106
12    4.693245
13    4.705026
9     4.718117
11    4.737462
22    4.738627
16    4.764790
10    4.796004
15    4.814754
14    4.843238
21    4.869057
Name: mutlak_hata, dtype: float64

✍️ *GFS gece mi, öğleden sonra mı daha iyi? En kötü saat kaç?*
öğleden sonra daha kötü. 21 en kötü saat

### Alıştırma 4 · Hafif rüzgârda tutarlı mı?

`sinif`'e göre grupla, `hata` ortalaması. 

⚠️ `sinif` bir *kategori* sütunu (`pd.cut` öyle üretiyor); groupby'a `observed=True` ekle, yoksa uyarı verir: `veri.groupby("sinif", observed=True)[...]`

In [29]:
veri.groupby("sinif",observed=True)["hata"].mean().sort_values()

sinif
kuvvetli   -10.550951
iyi         -8.336495
orta        -5.763235
hafif       -0.714757
Name: hata, dtype: float64

✍️ *Rüzgâr kuvvetlendikçe hata nasıl değişiyor? Hafifte GFS'e güvenilir mi?*
hafif rüzgarlarda gfs oldukça güvenilir rüzgar kuvvetlendikçe hata artıyıo

### Alıştırma 5 (isteğe bağlı) · Ortalama mı, tutarlılık mı?

"Tutarlı" demek yalnız ortalamanın sıfıra yakın olması değil, hataların da **az dağılması**. `.std()` (standart sapma) dağılmayı ölçer: küçükse hatalar hep birbirine yakın. 4'teki satırın sonunu `.std()` yap.

In [30]:
veri.groupby("sinif",observed=True)["hata"].std().sort_values()

sinif
hafif       2.580226
orta        3.091408
iyi         3.705985
kuvvetli    4.352506
Name: hata, dtype: float64

✍️ *Hangi sınıfta GFS en tutarlı?* hafif

## 3. Kapanış (2 cümle)

✍️ *groupby'ı kendi cümlemle anlatırsam:*
sütunları gruplandırmaya yarıyor. Oldukça pratik ve kullanışlı olduğunu düşünüyorum

✍️ *Bugünkü sonuçlar rüzgâr modeline ne söylüyor? (ipucu: model hangi durumlarda GFS'i en çok düzeltmek zorunda?)*
rüzgar modeline özellikle kuvvetli olanlarda çok hata yaptığını ve düzeltmesi gerektiğini söylüyor